# Trading Bot Skills Plugin - Validation Report

Automated validation of all 53 skills in the `trading-bot-skills` Claude Code plugin (v3.0.0).

**Checks performed:**
1. Frontmatter validity (YAML parse, required fields, name-directory match)
2. Cross-reference integrity (all referenced skills exist)
3. Config metadata consistency (version, skill count across files)
4. Doc reference validation (referenced docs exist)
5. Structural pattern coverage (Iron Law, Red Flags, Integration sections)
6. Companion file inventory

In [ ]:
import yaml
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path(".")
SKILLS_DIR = ROOT / "skills"
DOCS_DIR = ROOT / "docs"

# Discover all SKILL.md files
skill_files = sorted(SKILLS_DIR.glob("*/SKILL.md"))
skill_dirs = {f.parent.name for f in skill_files}

# Discover docs
doc_files = sorted(DOCS_DIR.glob("*.md"))
doc_names = {f.name for f in doc_files}

# Load config files
with open(ROOT / ".claude-plugin" / "plugin.json") as f:
    plugin_json = json.load(f)
with open(ROOT / ".claude-plugin" / "marketplace.json") as f:
    marketplace_json = json.load(f)
with open(ROOT / "package.json") as f:
    package_json = json.load(f)

print(f"Skills discovered: {len(skill_files)}")
print(f"Skill directories: {len(skill_dirs)}")
print(f"Doc files: {len(doc_files)} -> {sorted(doc_names)}")
print(f"Plugin version: {plugin_json.get('version')}")
print(f"Package version: {package_json.get('version')}")
print(f"Marketplace version: {marketplace_json.get('plugins', [{}])[0].get('version')}")
print()
if len(skill_files) == 53:
    print("[PASS] Expected 53 skills, found 53")
else:
    print(f"[FAIL] Expected 53 skills, found {len(skill_files)}")

In [ ]:
def parse_frontmatter(path):
    """Parse YAML frontmatter from a SKILL.md file. Returns (dict, errors)."""
    errors = []
    try:
        content = path.read_text(encoding="utf-8")
    except Exception as e:
        return None, [f"Cannot read file: {e}"]

    parts = content.split("---", 2)
    if len(parts) < 3:
        return None, ["Missing frontmatter delimiters (---)"]

    yaml_text = parts[1].strip()
    if not yaml_text:
        return None, ["Empty frontmatter block"]

    try:
        fm = yaml.safe_load(yaml_text)
    except yaml.YAMLError as e:
        return None, [f"YAML parse error: {e}"]

    if not isinstance(fm, dict):
        return None, [f"Frontmatter is not a dict: {type(fm)}"]

    if "name" not in fm:
        errors.append("Missing 'name' field")
    if "description" not in fm:
        errors.append("Missing 'description' field")

    return fm, errors


def validate_name_matches_dir(fm, dir_name):
    """Check if frontmatter name matches the skill directory name."""
    if fm and "name" in fm:
        if fm["name"] != dir_name:
            return f"Name mismatch: frontmatter='{fm['name']}' vs dir='{dir_name}'"
    return None


print("Frontmatter parser defined. Ready for validation.")